In [ ]:
%pip install -q pandas pyarrow matplotlib
import csv
import re
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
# Clone the repository into this Drive folder once, as described in the README.
ROOT = Path("/content/drive/MyDrive/Conversational-History-Retriever")
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
PAIRS_PATH = INTERIM / "pairs.parquet"
REWRITES_PATH = INTERIM / "rewrites.parquet"
for directory in (ROOT, INTERIM, PROCESSED, MODELS, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)
print(f"ROOT: {ROOT}")

In [ ]:
# Shared pairs contract: qid/query/did are nonempty strings; entry is ready to rank.
# Source title/url and additional columns are preserved. Splits are not assigned here.
def normalize_url(url):
    if pd.isna(url):
        raise ValueError("URL must not be missing or empty")
    url = str(url).strip()
    url = re.sub(r"^https?://", "", url, flags=re.IGNORECASE)
    url = re.sub(r"^www\.", "", url, flags=re.IGNORECASE)
    parts = re.split(r"([?#])", url, maxsplit=1)
    parts[0] = parts[0].rstrip("/")
    if not parts[0]:
        raise ValueError("URL must not be missing or empty")
    return "".join(parts)


def prepare_pairs(frame):
    required = {"qid", "query", "did", "title", "url"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Pairs table is missing required columns: {sorted(missing)}")
    pairs = frame.copy()
    for column in ("qid", "did", "query"):
        pairs[column] = pairs[column].astype("string")
        if (pairs[column].isna() | pairs[column].str.strip().eq("")).any():
            raise ValueError(f"Pairs column {column!r} contains missing or blank values")
    titles = pairs["title"].astype("string").fillna("").str.strip()
    urls = pairs["url"].map(normalize_url).astype("string")
    pairs["entry"] = (titles + " | " + urls).where(titles.ne(""), urls)
    if "split" in pairs:
        pairs["split"] = pairs["split"].replace({"validation": "dev"})
        if not pairs["split"].isin(["train", "dev", "test"]).all():
            raise ValueError("split values must be train, dev, or test (validation maps to dev)")
    return pairs

In [ ]:
# Small, explicit checks of formatting and source preservation.
url_examples = {
    "https://www.example.com/laptops/": "example.com/laptops",
    "http://example.com/": "example.com",
    "https://example.com/Review?id=12": "example.com/Review?id=12",
    "https://example.com/A/?next=/": "example.com/A?next=/",
    " HTTPS://WWW.example.com/A/#Section/ ": "example.com/A#Section/",
}
for original, expected in url_examples.items():
    assert normalize_url(original) == expected
sample = pd.DataFrame({
    "qid": ["001", "002", "003"], "query": ["one", "two", "three"],
    "did": ["01", "02", "03"], "title": ["  A review  ", None, "   "],
    "url": ["https://www.example.com/A/", "http://example.com/", "https://example.com/B/"],
})
prepared = prepare_pairs(sample)
assert prepared["entry"].tolist() == ["A review | example.com/A", "example.com", "example.com/B"]
pd.testing.assert_series_equal(prepared["url"], sample["url"])
pd.testing.assert_series_equal(prepared["title"], sample["title"])
print("Entry formatting checks passed")

In [ ]:
# Prefer the shared export; import the legacy dataset before rebuilding raw data.
legacy_pairs_path = ROOT / "data/data.parquet"
pairs_source = PAIRS_PATH if PAIRS_PATH.exists() else legacy_pairs_path
cached = pairs_source.exists()
RAW = Path("/content/raw")
if cached:
    df = pd.read_parquet(pairs_source)
    print(f"Loaded pairs from {pairs_source}")
else:
    RAW.mkdir(parents=True, exist_ok=True)
    !curl -fL -C - -o /content/raw/orcas.tsv.gz https://msmarco.z22.web.core.windows.net/msmarcoranking/orcas.tsv.gz
    !curl -fL -C - -o /content/raw/msmarco-docs.tsv.gz https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco-docs.tsv.gz


In [ ]:
# Join each click to its page title. QUOTE_NONE and keep_default_na stop queries like "null" from being misread.
if not cached:
    # Keep only the doc ID and title columns, so pandas never parses the ~22 GB of body text.
    !zcat /content/raw/msmarco-docs.tsv.gz | cut -f1,3 > /content/raw/doc_titles.tsv
    read = dict(sep="\t", dtype="string[pyarrow]", quoting=csv.QUOTE_NONE, keep_default_na=False)
    orcas = pd.read_csv("/content/raw/orcas.tsv.gz", names=["qid", "query", "did", "url"], **read)
    docs = pd.read_csv("/content/raw/doc_titles.tsv", names=["did", "title"], **read)
    docs = docs[docs["did"].isin(orcas["did"].unique())]
    df = orcas.merge(docs, on="did", validate="many_to_one")
    print(f"{len(orcas):,} clicks, {len(df):,} with a page title")
    del orcas, docs  # free memory before cleaning

In [ ]:
if not cached:
    df["query"] = df["query"].str.strip()
    df["title"] = df["title"].str.strip()
    df = df[df["query"].str.contains(r"[^\W_]") & df["title"].str.contains(r"[^\W_]")]
    # Titles over 20 words (about the top 1%) are mostly scraped page text, not real titles.
    df = df[df["title"].str.count(r"\S+") <= 20]
    df = df.drop_duplicates("qid")
    # Non-ASCII queries are mostly non-English, and Qwen tends to translate them instead of rewriting.
    df = df[~df["query"].str.contains(r"[^\x00-\x7F]")]
    # One query per page keeps any later split disjoint and avoids false in-batch negatives.
    df = df.drop_duplicates("did")
    df = df.sample(n=152_500, random_state=0).reset_index(drop=True)


In [ ]:
# Export before Qwen runs, so every retrieval notebook can consume the same table.
df = prepare_pairs(df)
temporary_pairs_path = PAIRS_PATH.with_suffix(".tmp.parquet")
df.to_parquet(temporary_pairs_path, index=False)
reloaded_pairs = pd.read_parquet(temporary_pairs_path)
pd.testing.assert_frame_equal(df.reset_index(drop=True), reloaded_pairs)
temporary_pairs_path.replace(PAIRS_PATH)
print(f"Saved {len(df):,} pairs to {PAIRS_PATH}")
display(df[["qid", "query", "did", "title", "url", "entry"]].head(10))

In [ ]:
words = pd.DataFrame({"query": df["query"].str.count(r"\S+"), "title": df["title"].str.count(r"\S+")}).astype(int)
site = df["url"].map(normalize_url).str.split(r"[/?#]", regex=True).str[0]
per_site = site.value_counts()
print(f"{len(df):,} query–page pairs from {len(per_site):,} sites")
print(f"Top site: {per_site.index[0]} ({per_site.iloc[0] / len(df):.1%} of pages)")
print(f"Sites with 200+ pages: {(per_site >= 200).sum()}, covering {per_site[per_site >= 200].sum() / len(df):.0%} of pages")
words.describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, col in zip(axes, ["query", "title"]):
    ax.hist(words[col].clip(upper=40), bins=range(1, 42), align="left", color="#2a78d6")
    ax.axvline(words[col].median(), color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{col.capitalize()} length (median {words[col].median():.0f} words)", loc="left")
    ax.set_xlabel("Words (40 = 40 or more)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Pairs")
plt.tight_layout()
plt.savefig(RESULTS / "lengths.png", dpi=200)
plt.show()

In [ ]:
top = per_site.head(15) / len(df) * 100
plt.figure(figsize=(8, 4.5))
plt.barh(top.index, top.values, color="#2a78d6")
plt.gca().invert_yaxis()
plt.gca().spines[["top", "right"]].set_visible(False)
plt.xlabel("% of pages")
plt.title("Top 15 sites in the dataset", loc="left")
for i, p in enumerate(top.values):
    plt.text(p + 0.3, i, f"{p:.1f}%", va="center")
plt.tight_layout()
plt.savefig(RESULTS / "top_sites.png", dpi=200)
plt.show()

In [ ]:
# Only the rows among the first N_REWRITE without a saved rewrite are generated.
N_REWRITE = 500
legacy_rewrites_path = ROOT / "data/rewrites.parquet"
rewrite_source = REWRITES_PATH if REWRITES_PATH.exists() else legacy_rewrites_path
if rewrite_source.exists():
    rewrites = pd.read_parquet(rewrite_source)
    rewrites["qid"] = rewrites["qid"].astype("string")
    if rewrite_source == legacy_rewrites_path:
        rewrites.to_parquet(REWRITES_PATH, index=False)
else:
    rewrites = pd.DataFrame({"qid": pd.Series(dtype="string"), "rewrite": [], "score": []})
to_rewrite = df.head(N_REWRITE)
to_rewrite = to_rewrite[~to_rewrite["qid"].isin(rewrites["qid"])]
print(f"{len(rewrites):,} rewrites saved, {len(to_rewrite):,} to go")


In [ ]:
%pip install -q transformers accelerate bitsandbytes
import random
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Qwen2.5-7B-Instruct already quantized to 4-bit: a 5.5 GB download instead of 15 GB, and no RAM spike from converting it.
MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
if len(to_rewrite) > 0:
    tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", torch_dtype=torch.float16)
    # The checkpoint computes in bfloat16, which T4s lack, so switch its 4-bit layers to float16.
    for layer in model.modules():
        if hasattr(layer, "compute_dtype"):
            layer.compute_dtype = torch.float16

In [ ]:
SYSTEM = """You rewrite search-engine queries into questions a person would ask an AI assistant that searches their own browsing history.

You will get the keyword query someone typed and the title of the page they clicked. Imagine that same person days later, trying to find that page again by asking the assistant in plain, everyday words.

Rules:
- Write one question or request, 6 to 20 words, in casual language.
- Refer to the page as something they already saw ("that article", "the site I used", "where I read about"), and vary how you do it.
- Keep the main name a person would remember, such as the site, show, brand or place, so the page can still be found. Describe the rest loosely, the way people do when half-remembering.
- Never copy three or more words in a row from the page title.
- Use only information in the query or the title. Do not invent details like dates, times, prices or reasons.
- Output only the question, with no quotes or explanation."""

EXAMPLES = [
    ("stranger things netflix", "Stranger Things | Netflix Official Site",
     "Pull up the Stranger Things page I had open on Netflix."),
    ("chicken thigh oven temp", "How Long to Bake Chicken Thighs at 400°F",
     "What temperature did that recipe say to bake chicken thighs at?"),
    ("irs w4 form", "About Form W-4, Employee's Withholding Certificate | Internal Revenue Service",
     "Where's the W-4 tax form I was filling out?"),
    ("how to tie a tie youtube", "How to Tie a Tie (Windsor Knot) - YouTube",
     "Find that YouTube video I watched on tying a tie."),
    ("list comprehension python", "Python - List Comprehension - GeeksforGeeks",
     "Find that Python tutorial I was reading on list comprehensions."),
    ("how many cups in a quart", "How Many Cups in a Quart? | Kitchen Conversions",
     "Find the page I used to convert quarts to cups."),
    ("symptoms of low iron", "Iron Deficiency Anemia: Symptoms, Causes & Treatment",
     "What was that article I read about signs of low iron?"),
    ("best budget laptops", "The 6 Best Budget Laptops - RTINGS.com",
     "Which cheap laptops was I looking at?"),
    ("zillow austin rentals", "Austin TX Apartments for Rent | Zillow",
     "Pull up those Austin apartments I was looking at on Zillow."),
]

def build_messages(qid, query, title):
    shots = random.Random(qid).sample(EXAMPLES, 3)
    messages = [{"role": "system", "content": SYSTEM}]
    for q, t, answer in shots:
        messages.append({"role": "user", "content": f"Query: {q}\nPage title: {t}"})
        messages.append({"role": "assistant", "content": answer})
    messages.append({"role": "user", "content": f"Query: {query}\nPage title: {title}"})
    return messages

In [ ]:
# Greedy decoding; score = average log-probability of the generated tokens (higher = Qwen was more confident).
@torch.no_grad()
def rewrite_batch(rows):
    prompts = [
        tokenizer.apply_chat_template(build_messages(r.qid, r.query, r.title), tokenize=False, add_generation_prompt=True)
        for r in rows
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**inputs, do_sample=False, max_new_tokens=40, output_scores=True, return_dict_in_generate=True)

    new_tokens = out.sequences[:, inputs["input_ids"].shape[1]:]
    logprobs = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
    mask = new_tokens != tokenizer.pad_token_id
    scores = torch.where(mask, logprobs, 0).sum(1) / mask.sum(1)

    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts], scores.tolist()

In [ ]:
# Save every 5,000 rewrites, so a crash partway through keeps the finished chunks.
for c in range(0, len(to_rewrite), 5_000):
    chunk = to_rewrite.iloc[c : c + 5_000]
    texts, scores = [], []
    for i in tqdm(range(0, len(chunk), 32)):
        t, s = rewrite_batch(list(chunk.iloc[i : i + 32].itertuples()))
        texts += t
        scores += s
    new = pd.DataFrame({"qid": chunk["qid"].values, "rewrite": texts, "score": scores})
    rewrites = pd.concat([rewrites, new], ignore_index=True)
    rewrites.to_parquet(REWRITES_PATH, index=False)

In [ ]:
# Other notebooks attach rewrites the same way; rows without one get empty columns.
data = df.merge(rewrites, on="qid", how="left")
data.loc[data["rewrite"].notna(), ["query", "title", "rewrite", "score"]].head(10)

In [ ]:
import re

# Check the rewrites against the prompt rules and the proposal's filters.
r = data[data["rewrite"].notna()].copy()
r["words"] = r["rewrite"].str.split().str.len()
r["opening"] = r["rewrite"].str.lower().str.split().str[:3].str.join(" ")

def copies_title(rewrite, title):
    t = re.findall(r"\w+", title.lower())
    w = " " + " ".join(re.findall(r"\w+", rewrite.lower())) + " "
    return any(" " + " ".join(t[i : i + 4]) + " " in w for i in range(len(t) - 3))

r["copies_title"] = [copies_title(a, b) for a, b in zip(r["rewrite"], r["title"])]

print(f"{len(r):,} rewrites")
print(f"Within 6-20 words: {r['words'].between(6, 20).mean():.0%}")
print(f"Copy 4+ title words in a row: {r['copies_title'].mean():.0%}")
print(f"Duplicate rewrites: {r['rewrite'].duplicated().sum()}")
r.sort_values("score")[["query", "title", "rewrite", "score"]].head(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].hist(r["score"], bins=30, color="#2a78d6")
axes[0].set_title("Rewrite confidence (higher is better)", loc="left")
axes[0].set_xlabel("Average log-probability")

axes[1].hist(r["words"].clip(upper=30), bins=range(1, 32), color="#2a78d6")
axes[1].axvspan(6, 21, color="gray", alpha=0.15)
axes[1].set_title("Rewrite length (shaded: 6-20 words)", loc="left")
axes[1].set_xlabel("Words")

top = r["opening"].value_counts().head(10) / len(r) * 100
axes[2].barh(top.index, top.values, color="#2a78d6")
axes[2].invert_yaxis()
axes[2].set_title("Most common openings", loc="left")
axes[2].set_xlabel("% of rewrites")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS / "rewrite_quality.png", dpi=200)
plt.show()

In [ ]:
# Share of pages from each of the top 10 sites: whole dataset vs the rows that were rewritten.
sites = pd.DataFrame({
    "Dataset": df["url"].map(normalize_url).str.split(r"[/?#]", regex=True).str[0].value_counts(normalize=True) * 100,
    "Rewritten rows": r["url"].map(normalize_url).str.split(r"[/?#]", regex=True).str[0].value_counts(normalize=True) * 100,
}).fillna(0).sort_values("Dataset", ascending=False).head(10)

ax = sites.plot.barh(color=["#2a78d6", "#eb6834"], figsize=(8, 4.5))
ax.invert_yaxis()
ax.set_xlabel("% of pages")
ax.set_ylabel("")
ax.set_title("Top sites: dataset vs rewritten rows", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(RESULTS / "rewrite_sites.png", dpi=200)
plt.show()

Shared Drive handoff: `data/interim/pairs.parquet` contains `qid`, `query`,
`did`, source `title` and `url`, and the canonical `entry` used by all three
retrieval notebooks. Any extra columns are preserved; this notebook does not
assign splits. `data/interim/rewrites.parquet` stores Qwen rewrites and confidence
scores, joined by string `qid`. Diagnostic plots are saved in `results/`.
These files stay in the shared Drive folder; no Git commit or push is needed.